<a href="https://colab.research.google.com/github/MrStranger812/MovieLens-CLI-DM/blob/NotebookGCPP/Notebooks/01_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Environment Cleanup
!pip uninstall -y cudf-cu11 dask-cudf-cu11 cuml-cu11 cugraph-cu11 cupy-cuda11x
!pip uninstall -y cudf-cu12 dask-cudf-cu12 cuml-cu12 cugraph-cu12 cupy-cuda12x

Found existing installation: cudf-cu12 25.6.0
Uninstalling cudf-cu12-25.6.0:
  Successfully uninstalled cudf-cu12-25.6.0
Found existing installation: dask-cudf-cu12 25.6.0
Uninstalling dask-cudf-cu12-25.6.0:
  Successfully uninstalled dask-cudf-cu12-25.6.0
Found existing installation: cuml-cu12 25.6.0
Uninstalling cuml-cu12-25.6.0:
  Successfully uninstalled cuml-cu12-25.6.0
Found existing installation: cupy-cuda12x 13.3.0
Uninstalling cupy-cuda12x-13.3.0:
  Successfully uninstalled cupy-cuda12x-13.3.0


In [ ]:
# Install version 25.6.0 to match your existing environment
!pip install --extra-index-url=https://pypi.nvidia.com 'cudf-cu12==25.6.0' 'dask-cudf-cu12==25.6.0' 'cuml-cu12==25.6.0' 'cugraph-cu12==25.6.0'

# Restart the kernel after installation
import os
os.kill(os.getpid(), 9)

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 66.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 69.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 194.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 228.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.9/112.9 MB 19.4 MB/s eta 0:00:00


In [1]:
import os
import sys
import gc
import psutil
import torch
import pandas as pd
import numpy as np
import pickle
import gzip
from pathlib import Path

# Import visualization and progress bar libraries
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box
from tqdm.notebook import tqdm

# --- Initial Setup & System Check ---
console = Console()

def system_check():
    """Checks for GPU and reports system status."""
    console.print(Panel.fit("[bold blue]🔧 System Resource Check[/bold blue]", border_style="blue"))
    gpu_available = False
    try:
        import cudf
        if torch.cuda.is_available():
            gpu_name = torch.cuda.get_device_name(0)
            gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
            console.print(f"[green]✅ GPU Available: {gpu_name} ({gpu_memory:.1f} GB)[/green]")
            gpu_available = True
        else:
            console.print("[bold red]❌ No GPU detected. This notebook requires a GPU runtime.[/bold red]")
    except Exception as e:
        console.print(f"[bold red]❌ Error during GPU check: {e}[/bold red]")

    ram_gb = psutil.virtual_memory().total / 1024**3
    console.print(f"[blue]💾 Available RAM: {ram_gb:.1f} GB[/blue]")
    return gpu_available

gpu_ready = system_check()

# Define project paths directly in the notebook
BASE_DIR = Path("/content/drive/MyDrive/Movielens")
DATA_DIR = BASE_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

╭──────────────────────────╮
│ 🔧 System Resource Check │
╰──────────────────────────╯

✅ GPU Available: NVIDIA A100-SXM4-40GB (39.6 GB)

💾 Available RAM: 83.5 GB

In [4]:
from sklearn.preprocessing import MultiLabelBinarizer
import cudf
from cuml.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix

class DataCleaner:
    """Handles loading and cleaning of MovieLens data."""
    def __init__(self):
        self.console = Console()
        self.raw_data_dir = RAW_DATA_DIR / "ml-20m"

    def load_data(self):
        self.console.print("[cyan]Loading MovieLens data...[/cyan]")
        try:
            ratings_df = pd.read_csv(self.raw_data_dir / "ratings.csv")
            movies_df = pd.read_csv(self.raw_data_dir / "movies.csv")
            tags_df = pd.read_csv(self.raw_data_dir / "tags.csv")
            return ratings_df, movies_df, tags_df
        except Exception as e:
            self.console.print(f"[red]Error loading data: {e}[/red]")
            raise

    def clean_ratings(self, ratings_df: pd.DataFrame) -> pd.DataFrame:
        self.console.print("[cyan]Cleaning ratings data...[/cyan]")
        ratings_df['timestamp'] = pd.to_datetime(ratings_df['timestamp'], unit='s')
        ratings_df['year'] = ratings_df['timestamp'].dt.year
        ratings_df['month'] = ratings_df['timestamp'].dt.month
        return ratings_df

    def clean_movies(self, movies_df: pd.DataFrame) -> pd.DataFrame:
        self.console.print("[cyan]Cleaning movies data...[/cyan]")
        movies_df['year'] = movies_df['title'].str.extract(r'\((\d{4})\)$', expand=False)
        movies_df['year'] = pd.to_numeric(movies_df['year'], errors='coerce')
        movies_df['clean_title'] = movies_df['title'].str.replace(r'\s*\(\d{4}\)$', '', regex=True)
        movies_df['genres'] = movies_df['genres'].str.split('|')
        return movies_df

class GPUFeatureTransformer:
    """Creates features using GPU acceleration with cuDF."""
    def __init__(self):
        self.console = Console()

    def create_user_features_gpu(self, ratings_gdf: cudf.DataFrame) -> cudf.DataFrame:
        self.console.print("[magenta]Creating user features on GPU...[/magenta]")
        user_stats = ratings_gdf.groupby('userId').agg({
            'rating': ['count', 'mean', 'std'],
            'movieId': 'nunique'
        })
        user_stats.columns = ['rating_count', 'rating_mean', 'rating_std', 'unique_movies_rated']
        user_stats['rating_std'] = user_stats['rating_std'].fillna(0)
        return user_stats.reset_index()

    def _binarize_genres_gpu(self, movies_gdf: cudf.DataFrame) -> cudf.DataFrame:
        """Helper function to one-hot encode genres on the GPU."""
        self.console.print("[magenta]Binarizing movie genres on GPU...[/magenta]")
        genres_exploded = movies_gdf[['movieId', 'genres']].explode('genres')
        genre_dummies = cudf.get_dummies(genres_exploded, columns=['genres'], prefix='', prefix_sep='')
        genre_features = genre_dummies.groupby('movieId').sum()
        return genre_features.reset_index()

    def create_movie_features_gpu(self, ratings_gdf: cudf.DataFrame, movies_gdf: cudf.DataFrame) -> cudf.DataFrame:
        self.console.print("[magenta]Creating movie features on GPU...[/magenta]")
        movie_stats = ratings_gdf.groupby('movieId').agg({
            'rating': ['count', 'mean', 'std']
        })
        movie_stats.columns = ['rating_count', 'rating_mean', 'rating_std']
        movie_stats['rating_std'] = movie_stats['rating_std'].fillna(0)
        genre_features = self._binarize_genres_gpu(movies_gdf)
        movies_with_genres = movies_gdf.merge(genre_features, on='movieId', how='left')
        movie_features = movies_with_genres.merge(movie_stats, on='movieId', how='left')
        feature_cols_to_fill = ['rating_count', 'rating_mean', 'rating_std']
        for col in feature_cols_to_fill:
            movie_features[col] = movie_features[col].fillna(0)
        genre_cols = [col for col in genre_features.columns if col != 'movieId']
        for col in genre_cols:
            movie_features[col] = movie_features[col].fillna(0).astype('int8')
        current_year = pd.to_datetime('today').year
        movie_features['movie_age'] = current_year - movie_features['year'].fillna(current_year)
        return movie_features

    def apply_pca_to_movies_gpu(self, movie_features_gdf: cudf.DataFrame, n_components: int = 32) -> cudf.DataFrame:
        """Applies PCA to reduce the dimensionality of movie features."""
        from cuml.decomposition import PCA

        exclude_cols = ['movieId', 'title', 'clean_title', 'genres', 'year']
        numeric_feature_cols = [col for col in movie_features_gdf.columns if col not in exclude_cols]
        movie_numeric_features = movie_features_gdf[numeric_feature_cols].astype('float32')

        # --- FIXED LOGIC ---
        # Dynamically adjust n_components to be no more than the number of available features.
        num_features = movie_numeric_features.shape[1]
        if n_components > num_features:
            self.console.print(f"[bold yellow]Warning:[/bold yellow] Requested {n_components} PCA components, but only {num_features} features are available. Adjusting to {num_features}.")
            n_components = num_features

        self.console.print(f"[magenta]Applying PCA to movie features (n_components={n_components})...[/magenta]")

        pca = PCA(n_components=n_components)
        movie_pca_features = pca.fit_transform(movie_numeric_features)
        movie_pca_features.columns = [f'pca_{i}' for i in range(n_components)]
        pca_result_gdf = cudf.concat([movie_features_gdf[['movieId']], movie_pca_features], axis=1)

        self.console.print(f"Explained variance from {n_components} components: {pca.explained_variance_ratio_.sum():.4f}")
        return pca_result_gdf

    def create_user_item_matrix(self, ratings_df: pd.DataFrame) -> (csr_matrix, dict, dict):
        self.console.print("[cyan]Creating user-item sparse matrix...[/cyan]")
        users = sorted(ratings_df['userId'].unique())
        movies = sorted(ratings_df['movieId'].unique())
        user_map = {user: i for i, user in enumerate(users)}
        movie_map = {movie: i for i, movie in enumerate(movies)}
        user_indices = ratings_df['userId'].map(user_map)
        movie_indices = ratings_df['movieId'].map(movie_map)
        matrix = csr_matrix((ratings_df['rating'], (user_indices, movie_indices)),
                            shape=(len(users), len(movies)))
        return matrix, user_map, movie_map

In [5]:
class GpuPreprocessingPipeline:
    """A self-contained pipeline that exclusively uses GPU acceleration."""
    def __init__(self):
        self.console = Console()
        self.cleaner = DataCleaner()
        self.transformer = GPUFeatureTransformer()

    def run(self, save_results: bool = True):
        self.console.print(Panel.fit("[bold magenta]🚀 MovieLens GPU Preprocessing Pipeline[/bold magenta]", border_style="magenta"))

        ratings_df, movies_df, _ = self.cleaner.load_data()
        ratings_df = self.cleaner.clean_ratings(ratings_df)
        movies_df = self.cleaner.clean_movies(movies_df)

        self.console.print("[magenta]Moving data to GPU...[/magenta]")
        ratings_gdf = cudf.from_pandas(ratings_df)
        movies_gdf = cudf.from_pandas(movies_df)

        # --- Feature Creation ---
        user_features_gdf = self.transformer.create_user_features_gpu(ratings_gdf)
        movie_features_gdf = self.transformer.create_movie_features_gpu(ratings_gdf, movies_gdf)
        user_item_matrix, user_map, movie_map = self.transformer.create_user_item_matrix(ratings_df)

        # --- NEW: Apply PCA to movie features ---
        movie_pca_features_gdf = self.transformer.apply_pca_to_movies_gpu(movie_features_gdf, n_components=32)

        # --- Convert results to Pandas for saving ---
        user_features_df = user_features_gdf.to_pandas()
        movie_features_df = movie_features_gdf.to_pandas()
        movie_pca_features_df = movie_pca_features_gdf.to_pandas()

        results = {
            'user_features': user_features_df,
            'movie_features': movie_features_df,
            'movie_pca_features': movie_pca_features_df,
            'user_item_matrix': user_item_matrix,
            'user_mapping': user_map,
            'movie_mapping': movie_map
        }

        if save_results: self.save_results(results)
        self.display_summary(results)
        return results

    def save_results(self, results: dict):
        self.console.print(f"[cyan]💾 Saving results to {PROCESSED_DATA_DIR}...[/cyan]")
        results['user_features'].to_parquet(PROCESSED_DATA_DIR / "user_features.parquet")
        results['movie_features'].to_parquet(PROCESSED_DATA_DIR / "movie_features.parquet")
        results['movie_pca_features'].to_parquet(PROCESSED_DATA_DIR / "movie_pca_features.parquet")

        from scipy.sparse import save_npz
        save_npz(PROCESSED_DATA_DIR / "user_item_matrix.npz", results['user_item_matrix'])

        with open(PROCESSED_DATA_DIR / "mappings.pkl", 'wb') as f:
            pickle.dump({'user_mapping': results['user_mapping'], 'movie_mapping': results['movie_mapping']}, f)
        self.console.print("[green]✅ Results saved successfully![/green]")

    def display_summary(self, results: dict):
        table = Table(title="📊 GPU Processing Summary", box=box.ROUNDED)
        table.add_column("Component", style="cyan")
        table.add_column("Details", style="yellow")
        table.add_row("User Features", f"{results['user_features'].shape[1]} features for {results['user_features'].shape[0]:,} users")
        table.add_row("Movie Features", f"{results['movie_features'].shape[1]} features for {results['movie_features'].shape[0]:,} movies")
        table.add_row("Movie PCA Features", f"{results['movie_pca_features'].shape[1]-1} components for {results['movie_pca_features'].shape[0]:,} movies")
        matrix = results['user_item_matrix']
        sparsity = 1 - (matrix.nnz / (matrix.shape[0] * matrix.shape[1]))
        table.add_row("User-Item Matrix", f"{matrix.shape[0]:,}x{matrix.shape[1]} (Sparsity: {sparsity:.4f})")
        self.console.print(table)

# --- Execute the Pipeline ---
if gpu_ready:
    console.print("[bold green]🔥 Starting Full GPU-Accelerated Processing...[/bold green]")
    gpu_pipeline = GpuPreprocessingPipeline()
    results = gpu_pipeline.run()
    if results:
        console.print("\n[bold green]🎉 GPU Preprocessing Complete! Ready for Machine Learning.[/bold green]")
else:
    console.print("\n[bold red]Pipeline execution halted. GPU not available.[/bold red]")

🔥 Starting Full GPU-Accelerated Processing...

╭─────────────────────────────────────────╮
│ 🚀 MovieLens GPU Preprocessing Pipeline │
╰─────────────────────────────────────────╯

Loading MovieLens data...

Cleaning ratings data...

Cleaning movies data...

...

Explained variance from 24 components: 1.0000

💾 Saving results to /content/drive/MyDrive/Movielens/data/processed...

✅ Results saved successfully!

                📊 GPU Processing Summary                
╭────────────────────┬──────────────────────────────────╮
│ Component          │ Details                          │
├────────────────────┼──────────────────────────────────┤
│ User Features      │ 5 features for 138,493 users     │
│ Movie Features     │ 29 features for 27,278 movies    │
│ Movie PCA Features │ 24 components for 27,278 movies  │
│ User-Item Matrix   │ 138,493x26744 (Sparsity: 0.9946) │
╰────────────────────┴──────────────────────────────────╯

🎉 GPU Preprocessing Complete! Ready for Machine Learning.